# 예제 04. 꽃 이미지 분류 실습
빅데이터프로그래밍 · 10주차

## 목표
- 꽃 이미지 데이터를 준비한다
- ResNet의 마지막 계층을 교체해 학습한다
- 예측 이미지·레이블·확률·틀린 이미지를 확인한다

적은 데이터로 얼마나 잘 되는지 직접 봅니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import pandas as pd
import time

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 내려받기
Flowers102 데이터셋의 일부를 씁니다. 클래스 5개만 골라 적은 데이터 상황을 만듭니다.


In [ ]:
IMG_SIZE = 224

# ImageNet 통계로 정규화 — 사전학습 모델이 이 값으로 학습됐습니다
NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    NORM,
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    NORM,
])

raw_train = datasets.Flowers102("./data", split="train", download=True, transform=train_tf)
raw_val   = datasets.Flowers102("./data", split="val",   download=True, transform=eval_tf)
print("원본 학습:", len(raw_train), "/ 검증:", len(raw_val))


In [ ]:
# 클래스 5개만 고릅니다
KEEP = [0, 1, 2, 3, 4]
NAMES = ["pink primrose", "hard-leaved orchid", "canterbury bells", "sweet pea", "english marigold"]

class Subset5(torch.utils.data.Dataset):
    def __init__(self, base):
        self.base = base
        self.idx = [i for i, lb in enumerate(base._labels) if lb in KEEP]
        self.remap = {c: i for i, c in enumerate(KEEP)}
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[self.idx[i]]
        return x, self.remap[y]


train_set = Subset5(raw_train)
val_set   = Subset5(raw_val)

N_CLASSES = len(KEEP)
print(f"학습 {len(train_set)}장 · 검증 {len(val_set)}장 · 클래스 {N_CLASSES}개")
print("→ 클래스당 학습 이미지가 10장 남짓입니다")


In [ ]:
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)

def show_batch(loader, n=8):
    x, y = next(iter(loader))
    inv = transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                               std=[1/0.229, 1/0.224, 1/0.225])
    fig, axes = plt.subplots(1, n, figsize=(16, 2.6))
    for ax, i in zip(axes, range(n)):
        ax.imshow(inv(x[i]).permute(1, 2, 0).clamp(0, 1))
        ax.set_title(NAMES[y[i]], fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

show_batch(train_loader)


## 2. 모델 — 마지막 계층만 교체하고 나머지는 고정


In [ ]:
def make_transfer(freeze=True, n_classes=N_CLASSES):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze:
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m


model = make_transfer().to(device)
total = sum(p.numel() for p in model.parameters())
train_n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 {total:,}개 중 학습 {train_n:,}개")
print("출력:", tuple(model(torch.randn(2, 3, 224, 224).to(device)).shape))


## 3. 학습


In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def train(model, epochs=10, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    start = time.time()
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append((*evaluate(model, train_loader), *evaluate(model, val_loader)))
        print(f"  epoch {epoch:2d}  학습 {hist[-1][1]:.4f}  검증 {hist[-1][3]:.4f}")
    return model, hist, time.time() - start


print("전이학습 (fc만 학습)")
model, hist, elapsed = train(model)
print(f"\n학습 시간 {elapsed:.1f}초 · 최종 검증 정확도 {hist[-1][3]:.4f}")


클래스당 10장으로 이 정확도가 나옵니다. 처음부터 학습시키면 불가능한 수준입니다.


In [ ]:
xs = range(1, len(hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(xs, [h[0] for h in hist], label="학습")
ax[0].plot(xs, [h[2] for h in hist], label="검증")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h[1] for h in hist], label="학습")
ax[1].plot(xs, [h[3] for h in hist], label="검증")
ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 4. 예측 결과 확인 — 이미지 · 정답 · 예측 · 확률


In [ ]:
inv = transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                           std=[1/0.229, 1/0.224, 1/0.225])

model.eval()
x, y = next(iter(val_loader))
with torch.no_grad():
    out = model(x.to(device))
    prob = torch.softmax(out, dim=1).cpu()
    pred = out.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(16, 5.6))
for ax, i in zip(axes.flatten(), range(12)):
    ax.imshow(inv(x[i]).permute(1, 2, 0).clamp(0, 1))
    ok = pred[i] == y[i]
    ax.set_title(f"{NAMES[pred[i]]}\n{prob[i, pred[i]]:.2f} / 정답 {NAMES[y[i]]}",
                 fontsize=8, color="green" if ok else "crimson")
    ax.axis("off")
plt.tight_layout(); plt.show()


## 5. 틀린 이미지만 모아 보기


In [ ]:
imgs, trues, preds, probs = [], [], [], []
with torch.no_grad():
    for x, y in val_loader:
        out = model(x.to(device))
        p = out.argmax(dim=1).cpu()
        pr = torch.softmax(out, dim=1).cpu()
        wrong = p != y
        if wrong.any():
            imgs.append(x[wrong]); trues.append(y[wrong])
            preds.append(p[wrong]); probs.append(pr[wrong].max(dim=1).values)

if imgs:
    imgs = torch.cat(imgs); trues = torch.cat(trues)
    preds = torch.cat(preds); probs = torch.cat(probs)
    n = min(8, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(2.2*n, 3))
    for ax, i in zip([axes] if n == 1 else axes, range(n)):
        ax.imshow(inv(imgs[i]).permute(1, 2, 0).clamp(0, 1))
        ax.set_title(f"{NAMES[preds[i]]} ({probs[i]:.2f})\n정답 {NAMES[trues[i]]}",
                     fontsize=8, color="crimson")
        ax.axis("off")
    plt.tight_layout(); plt.show()
    print("틀린 개수:", len(imgs))
else:
    print("전부 맞혔습니다")


## 6. 클래스별 정확도


In [ ]:
model.eval()
correct = torch.zeros(N_CLASSES); total = torch.zeros(N_CLASSES)
with torch.no_grad():
    for x, y in val_loader:
        p = model(x.to(device)).argmax(dim=1).cpu()
        for c in range(N_CLASSES):
            mask = y == c
            total[c] += mask.sum(); correct[c] += (p[mask] == c).sum()

pd.DataFrame({"클래스": NAMES,
              "개수": total.int().tolist(),
              "정확도": [round(v, 3) for v in (correct/total).tolist()]})


In [ ]:
torch.save(model.state_dict(), "flowers_transfer.pt")
print("저장 완료")


## 직접 해보기
1. 클래스를 10개로 늘리면 정확도가 어떻게 되나요?
2. `freeze=False` 로 전체 미세조정하면 (lr=1e-4) 결과가 어떻게 달라지나요?
3. 증강을 끄면 검증 정확도가 어떻게 되나요?


In [ ]:
# 여기에 작성하세요
